In [29]:
import os
import json
import chromadb

from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown
from chromadb.config import Settings

load_dotenv(override=True)



True

In [30]:
# chroma_client = chromadb.PersistentClient(path="./")

# collection = chroma_client.get_or_create_collection(name="restaurant_vector_db")

In [31]:
# llm_client = OpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=os.getenv("OPEN_ROUTER_KEY"),
# )

# # llm_client = OpenAI(
# #     base_url="https://api.groq.com/openai/v1",
# #     api_key=os.getenv("GROQ_API_KEY"),
# # )

# # MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free, NVIDIA)
# MODEL = "openai/gpt-oss-120b"
# EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b:free"

In [32]:
# Embeddings use NVIDIA's embedding model through OpenRouter.
# The chat model below remains Groq, as in the original notebook.
from getpass import getpass

OPEN_ROUTER_KEY = os.getenv("OPEN_ROUTER_KEY") or getpass("Paste your OpenRouter key (hidden): ")
embedding_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPEN_ROUTER_KEY,
)

def create_embeddings(texts):
    if not texts:
        return []
    response = embedding_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    return [item.embedding for item in response.data]


## Load + chunk

The same code as Day 30. The documents are in `chai_wala_docs`.

In [33]:
from pathlib import Path

def load_documents(folder):
    docs = []
    for path in sorted(Path(folder).glob("*.txt")):
        docs.append({
            "source": path.name,
            "text": path.read_text(encoding="utf-8")
        })
    return docs

def chunk_by_paragraph(text):
    paragraphs = []
    for paragraph in text.split("\n\n"):
        paragraph = paragraph.strip()
        if len(paragraph) > 30:  # skip short headings
            paragraphs.append(paragraph)
    return paragraphs

def chunk_documents(docs):
    chunks = []
    for doc in docs:
        for i, chunk_text in enumerate(chunk_by_paragraph(doc["text"])):
            chunks.append({
                "id": f"{doc['source']}-{i}",
                "source": doc["source"],
                "text": chunk_text
            })
    return chunks

documents = load_documents("chai_wala_docs")
chunks = chunk_documents(documents)
print(len(chunks), "paragraph chunks created")


0 paragraph chunks created


In [34]:
# Store paragraph chunks and their embeddings in persistent Chroma.
# The first run embeds and stores the chunks. Later runs reuse the saved vectors.
db = chromadb.PersistentClient(path="chai_wala_db")
collection = db.get_or_create_collection(
    name="restaurant_docs",
    metadata={"hnsw:space": "cosine"}
)

if collection.count() == 0 and chunks:
    chunk_ids = [chunk["id"] for chunk in chunks]
    chunk_texts = [chunk["text"] for chunk in chunks]
    chunk_metadatas = [{"source": chunk["source"]} for chunk in chunks]

    collection.add(
        ids=chunk_ids,
        documents=chunk_texts,
        metadatas=chunk_metadatas,
        embeddings=create_embeddings(chunk_texts)
    )

print(collection.count(), "chunks stored in Chroma")


0 chunks stored in Chroma


In [35]:
# Retrieve the 3 most relevant paragraph chunks for a customer question.
def retrieve(question, n=3):
    if collection.count() == 0:
        return []

    result = collection.query(
        query_embeddings=create_embeddings([question]),
        n_results=min(n, collection.count())
    )

    texts = result["documents"][0]
    metadatas = result["metadatas"][0]
    distances = result["distances"][0]

    found = []
    for i, chunk_text in enumerate(texts):
        found.append({
            "text": chunk_text,
            "source": metadatas[i].get("source", "unknown"),
            "distance": distances[i]
        })
    return found


def get_retrieval_context(question, n=3):
    found = retrieve(question, n=n)
    if not found:
        return "", []

    context_parts = []
    for item in found:
        context_parts.append(
            f"[Source: {item['source']} | Distance: {item['distance']:.4f}]\n{item['text']}"
        )
    return "\n\n".join(context_parts), found


# Optional inspection helper: see which chunks were retrieved.
def debug_retrieval(question):
    found = retrieve(question)
    print("Question:", question)
    for item in found:
        print(f"\n{item['distance']:.4f} [{item['source']}]\n{item['text']}")
    return found


In [36]:
menu = [
    {"name": "Zinger Burger", "price": 850},
    {"name": "Chicken Biryani", "price": 450},
    {"name": "Club Sandwich", "price": 700}
]

complaint_list = []

order_list = [
    "OR-1045",
    "OR-1046",
    "OR-1047"
]

In [37]:
def fetch_menu():
    return menu

def log_complaint(customer_name, order_id, phone, complaint):
    temp_dict = {
        "customer_name": customer_name,
        "order_id": order_id,
        "phone": phone,
        "complaint": complaint,
        "status": "Pending"
    }
    complaint_list.append(temp_dict)

    return {
        "message": f"Saved! Total Complaint: {len(complaint_list)}"
    }

def validate_order_id(order_id):
    if order_id in order_list:
        return True
    return False


tools = [
    {
        "type": "function",
        "function": {
                "name": "fetch_menu",
                "description": "Get the restaurant menu",
                # no inputs
                "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "validate_order_id",
            "description": "Validate order id before calling log_complaint tool it return True/False",
            "parameters": {
                "type": "object", 
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "the order id of which customer ordered food. OrderId should be like this OR-0000",
                    },
                },
                "required": ["order_id"],
                "additionalProperties": False,

            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "log_complaint",
            "description": "Log the complain",
            "strict": True,
            "parameters": {
                "type": "object", 
                "properties": {
                    "customer_name": {
                        "type": "string",
                        "description": "Customer name who is logging complain.",
                    },
                    "order_id": {
                        "type": "string",
                        "description": "the order id of which customer ordered food. OrderId should be like this OR-0000",
                    },
                    "phone": {
                        "type": "string",
                        "description": "Customer phone number or '-' ",
                    },
                    "complaint": {
                        "type": "string",
                        "description": "customer complain should me not more than 8 words",
                    }
                },
                "required": ["customer_name", "order_id", "phone", "complaint"],
                "additionalProperties": False,

            },
        }
    }

]

In [38]:
messages = [
    {
      "role": "system",  # User | Assistant | System
      "content": """
       You are the AI Customer Support Assistant for **Xander Restaurant**. Your job is to answer customer queries in a friendly, professional, and helpful way.

        ### Tools
        1. **`fetch_menu`** — Call whenever customers ask about menu items, prices, ingredients, or food recommendations. Use the tool's response to provide accurate information.
        2. **`log_complaint`** — Call whenever a customer raises a complaint. Collect the necessary details and log the complaint using this schema:

        ```python
        complaint = {
            "customer_name": "",
            "phone": "",
            "complaint": "",
            "order_id": "",
            "status": "Pending"
        }
        ```

        3. **`validate_order_id`** — Before logging complaint trigger this tool call validate_order_id to validate wether the order id is valid or not then call log_complaint


        Ask for missing necessary details before logging. The `order_id` can be empty if unavailable. Set the initial status to `"Pending"`. Confirm the complaint is logged only after the tool succeeds.

        ### Retrieval-Augmented Generation (RAG)
        - For questions about restaurant information, policies, services, ingredients, timings, delivery, or other facts documented in chai_wala_docs, use the retrieved context supplied with the customer question.
        - Treat retrieved context as the source of truth. Do not invent facts that are not present in the retrieved context.
        - If the context does not contain the answer, say politely that you do not have that information and offer to connect the customer with the restaurant team.
        - Menu questions must still call fetch_menu so prices and available items come from the live menu tool.
        - Complaint handling must still use validate_order_id before log_complaint.

        ### Guidelines
        - Keep replies short, natural, and friendly.
        - Never invent menu details, prices, or restaurant policies.
        - If you don't know something, politely say so.
        - Make every customer feel valued.
        - All prices are in Pakistani Rupees. Write them as "Rs 850".
      """
    }
]


In [39]:
function_names = {
    "fetch_menu": fetch_menu,
    "log_complaint": log_complaint,
    "validate_order_id": validate_order_id
}

def ask_assistant(question):
    # 1. RETRIEVE: find the 3 closest restaurant-document chunks.
    context, retrieved_chunks = get_retrieval_context(question)

    # 2. AUGMENT: attach the retrieved context to the customer's question.
    if context:
        augmented_question = (
            "Use the following retrieved restaurant-document context when it is relevant. "
            "The source labels are provided for traceability.\n\n"
            f"Retrieved context:\n{context}\n\n"
            f"Customer question: {question}"
        )
    else:
        augmented_question = question

    messages.append({
        "role": "user",
        "content": augmented_question
    })

    # 3. GENERATE: the LLM answers with RAG context and can still call restaurant tools.
    while True:
        completion = llm_client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            temperature=0
        )

        reply = completion.choices[0].message
        messages.append(reply)

        if not reply.tool_calls:
            return Markdown(reply.content or "")

        for call in reply.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = function_names[call.function.name](**args)
            print("Tool called:", call.function.name, args, result)
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result)
            })


In [40]:
# ask_assistant("i ordered burger but roll was served WTH!!! and the bill was made with burger, my name is Sam, OR-1045, phone is 0322342423")

In [41]:
ask_assistant("what are your delivery timings")

I’m sorry, I don’t have the delivery‑timing details on hand right now. Let me connect you with our team so they can provide the exact information you need.

In [42]:
print(json.dumps(complaint_list))

[]




Create a new tool validate_order to validate orderId when logging complaint
if order id does not gets validated say user please provider correct Order ID